## Resnet-50

#### Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader
import os
from tqdm import tqdm

In [ ]:
DATA_DIR = "dataset"
BATCH_SIZE = 32
NUM_EPOCHS_FREEZE = 5
NUM_EPOCHS_FINETUNE = 10
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
weights = ResNet50_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.meta["mean"],
                         std=weights.meta["std"])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.meta["mean"],
                         std=weights.meta["std"])
])


In [ ]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)


In [ ]:
model = resnet50(weights=weights)

In [ ]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False


In [ ]:
# Replace final layer for dataset classes
in_features = model.fc.in_features

model.fc = nn.Linear(in_features, num_classes)

In [ ]:
model = model.to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=LR
)

In [ ]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    running_loss = 0

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
def validate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

#### Training 1 (Freeze Backbone)

In [ ]:
for epoch in range(NUM_EPOCHS_FREEZE):
    loss = train_one_epoch(train_loader, epoch, NUM_EPOCHS_FREEZE)
    acc = validate(val_loader)

    print(f"Freeze Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")


#### Training 2 Fine-Tune

In [ ]:
for param in model.layer4.parameters():
    param.requires_grad = True

In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [ ]:
for epoch in range(NUM_EPOCHS_FINETUNE):
    loss = train_one_epoch(train_loader, epoch, NUM_EPOCHS_FINETUNE)
    acc = validate(val_loader)

    print(f"Finetune Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")

#### Save Model

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, "cassava_resnet50.pth")

In [ ]:
from PIL import Image

checkpoint = torch.load("cassava_resnet50.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

## ConvNeXt-S

#### Preparation


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision.models import convnext_small, ConvNeXt_Small_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
weights = ConvNeXt_Small_Weights.DEFAULT

train_transform = weights.transforms()
val_transform = weights.transforms()


In [ ]:
train_dataset = datasets.ImageFolder(
    "dataset/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "dataset/val",
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)


In [ ]:
model = convnext_small(weights=weights)


In [ ]:
in_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(in_features, num_classes)


#### Training 1

In [ ]:
for param in model.features.parameters():
    param.requires_grad = False


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3
)


In [ ]:
def train_epoch(loader, epoch, epochs):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
def validate(loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [ ]:
EPOCHS_STAGE1 = 5

for epoch in range(EPOCHS_STAGE1):

    loss = train_epoch(train_loader, epoch, EPOCHS_STAGE1)
    acc = validate(val_loader)

    print(f"Stage1 Epoch {epoch+1} | Loss {loss:.4f} | Val Acc {acc:.4f}")


#### Training 2

In [ ]:
for param in model.parameters():
    param.requires_grad = True


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=1e-5)


In [ ]:
EPOCHS_STAGE2 = 15

for epoch in range(EPOCHS_STAGE2):

    loss = train_epoch(train_loader, epoch, EPOCHS_STAGE2)
    acc = validate(val_loader)

    print(f"Stage2 Epoch {epoch+1} | Loss {loss:.4f} | Val Acc {acc:.4f}")


#### Save Model

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "class_names": class_names
}, "convnext_cassava.pth")


#### Prediction Sample

In [ ]:
checkpoint = torch.load("convnext_cassava.pth")

class_names = checkpoint["class_names"]

model.load_state_dict(checkpoint["model_state"])
model.eval()

def predict(image_tensor):

    image_tensor = image_tensor.to(device)

    with torch.no_grad():
        outputs = model(image_tensor.unsqueeze(0))
        pred_idx = outputs.argmax(1).item()

    return class_names[pred_idx]


## Swin-T

#### Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import swin_t, Swin_T_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from tqdm import tqdm

In [ ]:
DATA_DIR = "dataset"

BATCH_SIZE = 32
FREEZE_EPOCHS = 5
FINETUNE_EPOCHS = 10

LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
weights = Swin_T_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

In [ ]:
model = swin_t(weights=weights)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Replace Classification Head
in_features = model.head.in_features

model.head = nn.Linear(in_features, num_classes)

In [ ]:
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.head.parameters(),
    lr=LR
)

In [ ]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    running_loss = 0

    for images, labels in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
def validate(loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


#### Training 1 (Freeze Backbone)

In [ ]:
for epoch in range(FREEZE_EPOCHS):
    loss = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    acc = validate(val_loader)

    print(f"Freeze Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")

#### Training 2 Fine-Tuning

In [ ]:
# Unfreeze last block and head for fine-tuning
for param in model.features[-1].parameters():
    param.requires_grad = True

In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [ ]:
for epoch in range(FINETUNE_EPOCHS):
    loss = train_one_epoch(train_loader)
    acc = validate(val_loader)

    print(f"Finetune Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, "cassava_swint.pth")

#### Prediction Script only

In [ ]:
checkpoint = torch.load("cassava_swint.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


## Vit-B

#### Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from tqdm.notebook import tqdm

In [ ]:
DATA_DIR = "dataset"

BATCH_SIZE = 32
FREEZE_EPOCHS = 5
FINETUNE_EPOCHS = 10

LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
weights = ViT_B_16_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])


In [ ]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

In [ ]:
# Load pre-trained ViT model
model = vit_b_16(weights=weights)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
in_features = model.heads.head.in_features

model.heads.head = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.heads.head.parameters(),
    lr=LR
)

In [ ]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    running_loss = 0

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
def validate(loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

#### Training 1 (Freeze BackBone)

In [ ]:
for epoch in range(FREEZE_EPOCHS):
    loss = train_one_epoch(train_loader)
    acc = validate(val_loader)

    print(f"Freeze Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")

#### Training 2 (Fine-Tuning)

In [ ]:
for param in model.encoder.layers[-1].parameters():
    param.requires_grad = True

In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [ ]:
for epoch in range(FINETUNE_EPOCHS):
    loss = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    acc = validate(val_loader)

    print(f"Finetune Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, "cassava_vitb16.pth")

#### Prediction Sample

In [ ]:
checkpoint = torch.load("cassava_vitb16.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


## EfficientNet-v2

#### Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from tqdm import tqdm

In [ ]:
DATA_DIR = "dataset"

BATCH_SIZE = 32
FREEZE_EPOCHS = 5
FINETUNE_EPOCHS = 10

LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
weights = EfficientNet_V2_S_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])


In [ ]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


In [ ]:
model = efficientnet_v2_s(weights=weights)


In [ ]:
for param in model.parameters():
    param.requires_grad = False


In [ ]:
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=LR
)


In [ ]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    running_loss = 0
    
    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
def validate(loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


#### Training 1 (Freeze Backbone)

In [ ]:
for epoch in range(FREEZE_EPOCHS):
    loss = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    acc = validate(val_loader)

    print(f"Freeze Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")


#### Training 2 (Fine-Tuning)

In [ ]:
for param in model.features[-1].parameters():
    param.requires_grad = True


In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)


In [ ]:
for epoch in range(FINETUNE_EPOCHS):
    loss = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    acc = validate(val_loader)

    print(f"Finetune Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")


In [ ]:
# Save Model
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, "cassava_efficientnetv2.pth")


#### Prediction Sample

In [ ]:
checkpoint = torch.load("cassava_efficientnetv2.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


## MobileNet-v3

#### Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from tqdm import tqdm

In [ ]:
DATA_DIR = "dataset"

BATCH_SIZE = 32
FREEZE_EPOCHS = 5
FINETUNE_EPOCHS = 10

LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
weights = MobileNet_V3_Large_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])


In [ ]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


In [ ]:
model = mobilenet_v3_large(weights=weights)


In [ ]:
for param in model.parameters():
    param.requires_grad = False


In [ ]:
in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=LR
)


In [ ]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    running_loss = 0

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
def validate(loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


#### Training 1 (Freeze Backbone)

In [ ]:
for epoch in range(FREEZE_EPOCHS):
    loss = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    acc = validate(val_loader)

    print(f"Freeze Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")


#### Training 2 (Fine-Tuning)

In [ ]:
for param in model.features[-1].parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)


In [ ]:
for epoch in range(FINETUNE_EPOCHS):
    loss = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    acc = validate(val_loader)

    print(f"Finetune Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")


In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, "cassava_mobilenetv3.pth")


#### Prediction Sample

In [ ]:
checkpoint = torch.load("cassava_mobilenetv3.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


## DenseNet121

#### Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import densenet121, DenseNet121_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from tqdm import tqdm

In [ ]:
DATA_DIR = "dataset"

BATCH_SIZE = 32
EPOCHS = 15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
weights = DenseNet121_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"],
        std=weights.meta["std"]
    )
])


In [ ]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


In [ ]:
model = densenet121(weights=weights)

in_features = model.classifier.in_features

model.classifier = nn.Linear(in_features, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_epoch(loader, epoch, epochs):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


#### Progressive Unfreezing

##### Training 1

In [ ]:
for param in model.features.parameters():
    param.requires_grad = False


In [ ]:
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)


In [ ]:
for epoch in range(EPOCHS):

    train_loss = train_one_epoch(train_loader)
    val_acc = validate(val_loader)

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")


##### Training 2

In [ ]:
for name, param in model.features.named_parameters():
    if "denseblock4" in name:
        param.requires_grad = True


In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)


In [ ]:
for epoch in range(EPOCHS):

    train_loss = train_one_epoch(train_loader)
    val_acc = validate(val_loader)

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")


##### Training 3

In [ ]:
for name, param in model.features.named_parameters():
    if "denseblock3" in name:
        param.requires_grad = True


In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)


In [ ]:
for epoch in range(EPOCHS):

    train_loss = train_one_epoch(train_loader)
    val_acc = validate(val_loader)

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")


##### Training 4 (Fine-Tuning)

In [ ]:
for param in model.parameters():
    param.requires_grad = True


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-5)


In [ ]:
for epoch in range(EPOCHS):

    train_loss = train_one_epoch(train_loader)
    val_acc = validate(val_loader)

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")


#### Save Model

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, "cassava_densenet121.pth")


#### Prediction Sample

In [ ]:
checkpoint = torch.load("cassava_densenet121.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

## YOLOv8

In [ ]:
from ultralytics import YOLO

In [ ]:
EPOCHS = 100
BATCH_SIZE = 16
Data_Dir = '/content/data.yaml'
Image_Size = 640

In [ ]:
model = YOLO('yolov8x.pt')

In [ ]:
train_results = model.train(
    data=Data_Dir,
    epochs=EPOCHS,
    imgsz=Image_Size,
    batch=BATCH_SIZE,
    name='YOLO_Cassava_Leaf'
)


## Crete YAML File